# SQL Concepts with SAP HANA Cloud using Python `hdbcli`

This notebook is designed for beginners who want to learn SQL concepts by connecting Python to **SAP HANA Cloud** using the `hdbcli` library.

You will learn:

1. How to connect Python to SAP HANA Cloud
2. How to create tables
3. How to insert data
4. How to read data using `SELECT`
5. How to filter, sort, and aggregate data
6. How to use `GROUP BY`
7. How to use joins
8. How to update and delete records
9. How to create a view
10. How to close the database connection safely

> Replace the database credentials with your own SAP HANA Cloud details before running the notebook.


## 1. Install and Import Required Library

`hdbcli` is the official Python client used to connect Python programs with SAP HANA.

Run the installation cell only once in your environment.


In [3]:
# Run this cell only if hdbcli is not installed
# In Jupyter Notebook, uncomment the next line:
!pip install hdbcli

from hdbcli import dbapi


## 2. Connect to SAP HANA Cloud

From your SAP HANA Database Explorer properties screen, you identified:

```text
Host = 13b7c15d-848f-40b5-9259-c9c36ab85f56.hna1.prod-eu10.hanacloud.ondemand.com
Port = 443
Database = H00
```

For `hdbcli`, normally you need:

- `address`: SAP HANA Cloud host name
- `port`: usually `443`
- `user`: database user, for example `DBADMIN`
- `password`: your database password
- `encrypt=True`: required for SAP HANA Cloud


In [4]:
# Update these values before running

HANA_HOST = "13b7c15d-848f-40b5-9259-c9c36ab85f56.hna1.prod-eu10.hanacloud.ondemand.com"
HANA_PORT = 443
HANA_USER = "GE336872"
HANA_PASSWORD = "ObxiYJJCSh1!"

conn = dbapi.connect(
    address=HANA_HOST,
    port=HANA_PORT,
    user=HANA_USER,
    password=HANA_PASSWORD,
    encrypt=True,
    sslValidateCertificate=False  # For training/demo. Use certificate validation in production.
)

cursor = conn.cursor()

print("Connected successfully to SAP HANA Cloud")


Connected successfully to SAP HANA Cloud


## 3. Test the Connection

`DUMMY` is a special one-row table in SAP HANA.  
It is commonly used to test SQL queries.


In [5]:
cursor.execute("SELECT CURRENT_USER, CURRENT_SCHEMA FROM DUMMY")
result = cursor.fetchone()

print("Current User:", result[0])
print("Current Schema:", result[1])


Current User: GE336872
Current Schema: GE336872


## 4. Create Tables

We will create three tables:

### `STUDENTS`
Stores student information.

### `COURSES`
Stores course information.

### `ENROLLMENTS`
Connects students with courses.

This is a common relational database design:

```text
STUDENTS  ----<  ENROLLMENTS  >----  COURSES
```

One student can enroll in many courses.  
One course can have many students.


In [6]:
# Drop existing tables if they already exist
# We drop child table first because it has foreign keys.

tables_to_drop = ["ENROLLMENTS", "STUDENTS", "COURSES"]

for table in tables_to_drop:
    try:
        cursor.execute(f"DROP TABLE {table}")
        print(f"Dropped table: {table}")
    except Exception as e:
        print(f"Table {table} does not exist or cannot be dropped. Skipping.")

conn.commit()


Table ENROLLMENTS does not exist or cannot be dropped. Skipping.
Table STUDENTS does not exist or cannot be dropped. Skipping.
Table COURSES does not exist or cannot be dropped. Skipping.


In [7]:
# Create STUDENTS table

cursor.execute("""
CREATE COLUMN TABLE STUDENTS (
    STUDENT_ID INTEGER PRIMARY KEY,
    STUDENT_NAME NVARCHAR(100) NOT NULL,
    CITY NVARCHAR(50),
    AGE INTEGER
)
""")

# Create COURSES table

cursor.execute("""
CREATE COLUMN TABLE COURSES (
    COURSE_ID INTEGER PRIMARY KEY,
    COURSE_NAME NVARCHAR(100) NOT NULL,
    FEES DECIMAL(10,2)
)
""")

# Create ENROLLMENTS table
# This table uses a composite primary key: STUDENT_ID + COURSE_ID

cursor.execute("""
CREATE COLUMN TABLE ENROLLMENTS (
    STUDENT_ID INTEGER NOT NULL,
    COURSE_ID INTEGER NOT NULL,
    ENROLLMENT_DATE DATE,
    MARKS INTEGER,
    PRIMARY KEY (STUDENT_ID, COURSE_ID),
    FOREIGN KEY (STUDENT_ID) REFERENCES STUDENTS(STUDENT_ID),
    FOREIGN KEY (COURSE_ID) REFERENCES COURSES(COURSE_ID)
)
""")

conn.commit()

print("Tables created successfully")


Tables created successfully


## 5. SQL Concept: Primary Key, Foreign Key, and Composite Key

### Primary Key
A primary key uniquely identifies each record in a table.

Example:

```sql
STUDENT_ID INTEGER PRIMARY KEY
```

This means two students cannot have the same `STUDENT_ID`.

### Foreign Key
A foreign key creates a relationship between two tables.

Example:

```sql
FOREIGN KEY (STUDENT_ID) REFERENCES STUDENTS(STUDENT_ID)
```

This means an enrollment can only be created for a student that already exists in the `STUDENTS` table.

### Composite Key
A composite key is made from more than one column.

Example:

```sql
PRIMARY KEY (STUDENT_ID, COURSE_ID)
```

This means the same student cannot be enrolled in the same course twice.


## 6. Insert Data into Tables

`INSERT INTO` is used to add records into a table.

We will insert:

- 5 students
- 4 courses
- multiple enrollments


In [8]:
# Insert data into STUDENTS

students = [
    (1, "Amit Kumar", "Patna", 22),
    (2, "Priya Singh", "Delhi", 21),
    (3, "Rahul Sharma", "Patna", 23),
    (4, "Neha Gupta", "Kolkata", 20),
    (5, "Saurabh Verma", "Bangalore", 24)
]

cursor.executemany(
    "INSERT INTO STUDENTS (STUDENT_ID, STUDENT_NAME, CITY, AGE) VALUES (?, ?, ?, ?)",
    students
)

# Insert data into COURSES

courses = [
    (101, "SQL Basics", 5000.00),
    (102, "Python for Data Analysis", 8000.00),
    (103, "SAP HANA Cloud", 12000.00),
    (104, "Data Visualization", 7000.00)
]

cursor.executemany(
    "INSERT INTO COURSES (COURSE_ID, COURSE_NAME, FEES) VALUES (?, ?, ?)",
    courses
)

# Insert data into ENROLLMENTS

enrollments = [
    (1, 101, "2026-07-01", 85),
    (1, 103, "2026-07-02", 78),
    (2, 101, "2026-07-01", 90),
    (2, 102, "2026-07-03", 88),
    (3, 103, "2026-07-04", 72),
    (4, 104, "2026-07-05", 95),
    (5, 102, "2026-07-06", 80),
    (5, 103, "2026-07-06", 83)
]

cursor.executemany(
    "INSERT INTO ENROLLMENTS (STUDENT_ID, COURSE_ID, ENROLLMENT_DATE, MARKS) VALUES (?, ?, ?, ?)",
    enrollments
)

conn.commit()

print("Sample data inserted successfully")


Sample data inserted successfully


## 7. Helper Function to Display Query Results

This helper function runs a SQL query and prints the result in a readable format.


In [9]:
def run_query(sql):
    cursor.execute(sql)
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]

    print(" | ".join(columns))
    print("-" * 80)

    for row in rows:
        print(" | ".join(str(value) for value in row))


## 8. SELECT: Read Data from a Table

`SELECT` is used to read data from a database table.


In [10]:
run_query("SELECT * FROM STUDENTS")

STUDENT_ID | STUDENT_NAME | CITY | AGE
--------------------------------------------------------------------------------
1 | Amit Kumar | Patna | 22
2 | Priya Singh | Delhi | 21
3 | Rahul Sharma | Patna | 23
4 | Neha Gupta | Kolkata | 20
5 | Saurabh Verma | Bangalore | 24


## 9. Select Specific Columns

Instead of selecting all columns using `*`, you can select only the columns you need.


In [11]:
run_query("""
SELECT STUDENT_ID, STUDENT_NAME, CITY
FROM STUDENTS
""")

STUDENT_ID | STUDENT_NAME | CITY
--------------------------------------------------------------------------------
1 | Amit Kumar | Patna
2 | Priya Singh | Delhi
3 | Rahul Sharma | Patna
4 | Neha Gupta | Kolkata
5 | Saurabh Verma | Bangalore


## 10. WHERE: Filter Records

`WHERE` is used to apply conditions.

Example: show only students from Patna.


In [12]:
run_query("""
SELECT *
FROM STUDENTS
WHERE CITY = 'Patna'
""")

STUDENT_ID | STUDENT_NAME | CITY | AGE
--------------------------------------------------------------------------------
1 | Amit Kumar | Patna | 22
3 | Rahul Sharma | Patna | 23


## 11. ORDER BY: Sort Records

`ORDER BY` is used to sort results.

Example: show students sorted by age.


In [13]:
run_query("""
SELECT *
FROM STUDENTS
ORDER BY AGE DESC
""")

STUDENT_ID | STUDENT_NAME | CITY | AGE
--------------------------------------------------------------------------------
5 | Saurabh Verma | Bangalore | 24
3 | Rahul Sharma | Patna | 23
1 | Amit Kumar | Patna | 22
2 | Priya Singh | Delhi | 21
4 | Neha Gupta | Kolkata | 20


## 12. Aggregate Functions

Aggregate functions calculate summary values.

Common aggregate functions:

| Function | Meaning |
|---|---|
| `COUNT()` | Counts records |
| `SUM()` | Adds values |
| `AVG()` | Calculates average |
| `MIN()` | Finds minimum |
| `MAX()` | Finds maximum |


In [14]:
run_query("""
SELECT
    COUNT(*) AS TOTAL_STUDENTS,
    AVG(AGE) AS AVERAGE_AGE,
    MIN(AGE) AS MIN_AGE,
    MAX(AGE) AS MAX_AGE
FROM STUDENTS
""")

TOTAL_STUDENTS | AVERAGE_AGE | MIN_AGE | MAX_AGE
--------------------------------------------------------------------------------
5 | 22 | 20 | 24


## 13. GROUP BY: Group Records

`GROUP BY` groups records based on one or more columns.

Example: count students city-wise.


In [15]:
run_query("""
SELECT
    CITY,
    COUNT(*) AS TOTAL_STUDENTS
FROM STUDENTS
GROUP BY CITY
ORDER BY TOTAL_STUDENTS DESC
""")

CITY | TOTAL_STUDENTS
--------------------------------------------------------------------------------
Patna | 2
Delhi | 1
Kolkata | 1
Bangalore | 1


## 14. HAVING: Filter Grouped Results

`WHERE` filters rows before grouping.  
`HAVING` filters groups after grouping.

Example: show cities where more than one student is present.


In [16]:
run_query("""
SELECT
    CITY,
    COUNT(*) AS TOTAL_STUDENTS
FROM STUDENTS
GROUP BY CITY
HAVING COUNT(*) > 1
""")

CITY | TOTAL_STUDENTS
--------------------------------------------------------------------------------
Patna | 2


## 15. INNER JOIN

`INNER JOIN` returns only matching records from both tables.

Example: show student names with their enrolled courses.


In [17]:
run_query("""
SELECT
    S.STUDENT_ID,
    S.STUDENT_NAME,
    C.COURSE_NAME,
    E.ENROLLMENT_DATE,
    E.MARKS
FROM ENROLLMENTS E
INNER JOIN STUDENTS S
    ON E.STUDENT_ID = S.STUDENT_ID
INNER JOIN COURSES C
    ON E.COURSE_ID = C.COURSE_ID
ORDER BY S.STUDENT_ID
""")

STUDENT_ID | STUDENT_NAME | COURSE_NAME | ENROLLMENT_DATE | MARKS
--------------------------------------------------------------------------------
1 | Amit Kumar | SQL Basics | 2026-07-01 | 85
1 | Amit Kumar | SAP HANA Cloud | 2026-07-02 | 78
2 | Priya Singh | SQL Basics | 2026-07-01 | 90
2 | Priya Singh | Python for Data Analysis | 2026-07-03 | 88
3 | Rahul Sharma | SAP HANA Cloud | 2026-07-04 | 72
4 | Neha Gupta | Data Visualization | 2026-07-05 | 95
5 | Saurabh Verma | Python for Data Analysis | 2026-07-06 | 80
5 | Saurabh Verma | SAP HANA Cloud | 2026-07-06 | 83


## 16. LEFT JOIN

`LEFT JOIN` returns all records from the left table and matching records from the right table.

Example: show all students and their courses if available.


In [18]:
run_query("""
SELECT
    S.STUDENT_ID,
    S.STUDENT_NAME,
    C.COURSE_NAME
FROM STUDENTS S
LEFT JOIN ENROLLMENTS E
    ON S.STUDENT_ID = E.STUDENT_ID
LEFT JOIN COURSES C
    ON E.COURSE_ID = C.COURSE_ID
ORDER BY S.STUDENT_ID
""")

STUDENT_ID | STUDENT_NAME | COURSE_NAME
--------------------------------------------------------------------------------
1 | Amit Kumar | SQL Basics
1 | Amit Kumar | SAP HANA Cloud
2 | Priya Singh | SQL Basics
2 | Priya Singh | Python for Data Analysis
3 | Rahul Sharma | SAP HANA Cloud
4 | Neha Gupta | Data Visualization
5 | Saurabh Verma | Python for Data Analysis
5 | Saurabh Verma | SAP HANA Cloud


## 17. UPDATE: Modify Existing Data

`UPDATE` is used to change existing records.

Example: update the marks of one student in one course.


In [19]:
cursor.execute("""
UPDATE ENROLLMENTS
SET MARKS = 92
WHERE STUDENT_ID = 1
  AND COURSE_ID = 101
""")

conn.commit()

print("Record updated successfully")

run_query("""
SELECT *
FROM ENROLLMENTS
WHERE STUDENT_ID = 1
  AND COURSE_ID = 101
""")


Record updated successfully
STUDENT_ID | COURSE_ID | ENROLLMENT_DATE | MARKS
--------------------------------------------------------------------------------
1 | 101 | 2026-07-01 | 92


## 18. DELETE: Remove Records

`DELETE` removes records from a table.

For safety, always use a `WHERE` condition.


In [20]:
# Example: delete one enrollment record

cursor.execute("""
DELETE FROM ENROLLMENTS
WHERE STUDENT_ID = 4
  AND COURSE_ID = 104
""")

conn.commit()

print("Record deleted successfully")

run_query("SELECT * FROM ENROLLMENTS ORDER BY STUDENT_ID, COURSE_ID")


Record deleted successfully
STUDENT_ID | COURSE_ID | ENROLLMENT_DATE | MARKS
--------------------------------------------------------------------------------
1 | 101 | 2026-07-01 | 92
1 | 103 | 2026-07-02 | 78
2 | 101 | 2026-07-01 | 90
2 | 102 | 2026-07-03 | 88
3 | 103 | 2026-07-04 | 72
5 | 102 | 2026-07-06 | 80
5 | 103 | 2026-07-06 | 83


## 19. Create a View

A view is a saved SQL query.

It does not store data separately like a table.  
It helps simplify complex queries.

Example: create a view that shows student-course enrollment details.


In [21]:
# Drop view if it already exists

try:
    cursor.execute("DROP VIEW STUDENT_COURSE_VIEW")
    conn.commit()
    print("Old view dropped.")
except Exception:
    print("View does not exist. Creating new view.")

cursor.execute("""
CREATE VIEW STUDENT_COURSE_VIEW AS
SELECT
    S.STUDENT_ID,
    S.STUDENT_NAME,
    S.CITY,
    C.COURSE_ID,
    C.COURSE_NAME,
    C.FEES,
    E.ENROLLMENT_DATE,
    E.MARKS
FROM ENROLLMENTS E
INNER JOIN STUDENTS S
    ON E.STUDENT_ID = S.STUDENT_ID
INNER JOIN COURSES C
    ON E.COURSE_ID = C.COURSE_ID
""")

conn.commit()

print("View created successfully")


View does not exist. Creating new view.
View created successfully


In [22]:
run_query("""
SELECT *
FROM STUDENT_COURSE_VIEW
ORDER BY STUDENT_ID, COURSE_ID
""")

STUDENT_ID | STUDENT_NAME | CITY | COURSE_ID | COURSE_NAME | FEES | ENROLLMENT_DATE | MARKS
--------------------------------------------------------------------------------
1 | Amit Kumar | Patna | 101 | SQL Basics | 5000 | 2026-07-01 | 92
1 | Amit Kumar | Patna | 103 | SAP HANA Cloud | 12000 | 2026-07-02 | 78
2 | Priya Singh | Delhi | 101 | SQL Basics | 5000 | 2026-07-01 | 90
2 | Priya Singh | Delhi | 102 | Python for Data Analysis | 8000 | 2026-07-03 | 88
3 | Rahul Sharma | Patna | 103 | SAP HANA Cloud | 12000 | 2026-07-04 | 72
5 | Saurabh Verma | Bangalore | 102 | Python for Data Analysis | 8000 | 2026-07-06 | 80
5 | Saurabh Verma | Bangalore | 103 | SAP HANA Cloud | 12000 | 2026-07-06 | 83


## 20. Practice Questions

Try writing SQL queries for the following:

1. Show all students whose age is greater than 21.
2. Show total enrollment count for each course.
3. Show average marks for each course.
4. Show students who enrolled in `SAP HANA Cloud`.
5. Show the highest marks scored by each student.
6. Create a view for course-wise performance.


## 21. Close the Connection

Always close the cursor and connection after your work is done.


In [23]:
cursor.close()
conn.close()

print("Connection closed successfully")


Connection closed successfully


2.	Hands-on Problem Statement: Hospital OPD Appointment and Billing Analytics
Domain
Healthcare / Hospital Management
Business Scenario
A hospital wants to manage and analyze its OPD appointments, doctor consultations, patient visits, departments, and billing information.
Currently, the hospital stores patient, doctor, department, appointment, and billing data separately. The hospital administration wants a consolidated reporting view to understand:
•	Which patients visited which doctor
•	Which department handled the consultation
•	What consultation fee was charged
•	What was the payment status
•	Which appointments are completed, cancelled, or pending
•	Department-wise and doctor-wise revenue analysis
Your task is to create a new database area/schema and design the required tables. After that, you need to create a SQL view that combines data from multiple tables and provides a meaningful business report.
________________________________________
Main Objective
Create a new database/schema named:
HOSPITAL_OPD_DB
Inside this database/schema, create the required tables and then create a view named:
V_OPD_APPOINTMENT_ANALYTICS
The view should provide a complete OPD appointment and billing summary.
________________________________________
Required Tables
1. PATIENTS
This table should store basic patient information.
Required information:
•	Patient ID
•	Patient Name
•	Gender
•	Age
•	City
•	Mobile Number
Purpose:
This table will help identify which patient visited the hospital and from which location.
________________________________________
2. DEPARTMENTS
This table should store hospital department details.
Required information:
•	Department ID
•	Department Name
•	Floor Number
Purpose:
This table will help categorize doctors and appointments department-wise, such as Cardiology, Orthopedics, General Medicine, Pediatrics, etc.
________________________________________
3. DOCTORS
This table should store doctor details.
Required information:
•	Doctor ID
•	Doctor Name
•	Department ID
•	Specialization
•	Consultation Fee
Purpose:
This table will help identify which doctor handled the consultation and what fee is charged for the visit.
________________________________________
4. APPOINTMENTS
This table should store appointment booking details.
Required information:
•	Appointment ID
•	Patient ID
•	Doctor ID
•	Appointment Date
•	Appointment Time
•	Appointment Status
Appointment Status can be:
•	Completed
•	Pending
•	Cancelled
Purpose:
This table will help track patient appointments with doctors.
________________________________________
5. BILLING
This table should store billing and payment details.
Required information:
•	Bill ID
•	Appointment ID
•	Bill Amount
•	Payment Mode
•	Payment Status
Payment Mode can be:
•	Cash
•	UPI
•	Card
•	Insurance
Payment Status can be:
•	Paid
•	Unpaid
•	Refunded
Purpose:
This table will help analyze hospital OPD revenue and payment status.
________________________________________
Relationship Between Tables
The database should follow these relationships:
1.	One patient can have multiple appointments.
2.	One doctor can handle multiple appointments.
3.	One department can have multiple doctors.
4.	One appointment can have one billing record.
5.	Appointment data should connect patients and doctors.
6.	Billing data should connect with appointment data.
________________________________________
Main View Requirement
Create a view named:
V_OPD_APPOINTMENT_ANALYTICS
The view should combine data from all related tables and display a consolidated report.
The view should include the following information:
Field Name	Description
Appointment ID	Unique appointment number
Appointment Date	Date of appointment
Appointment Time	Time of appointment
Patient Name	Name of the patient
Patient City	City of the patient
Doctor Name	Name of the doctor
Department Name	Hospital department name
Specialization	Doctor specialization
Consultation Fee	Doctor consultation fee
Bill Amount	Final billed amount
Payment Mode	Cash, UPI, Card, or Insurance
Payment Status	Paid, Unpaid, or Refunded
Appointment Status	Completed, Pending, or Cancelled
________________________________________
Business Questions to Answer Using the View
After creating the view, students should be able to answer the following questions:
Question 1
Find all completed OPD appointments.
Expected output should show:
•	Appointment ID
•	Patient Name
•	Doctor Name
•	Department Name
•	Appointment Date
•	Bill Amount
•	Payment Status
________________________________________
Question 2
Find total OPD revenue generated by the hospital.
Only appointments with Paid payment status should be considered.
________________________________________
Question 3
Find department-wise revenue.
Expected output should show:
•	Department Name
•	Total Revenue
________________________________________
Question 4
Find doctor-wise consultation revenue.
Expected output should show:
•	Doctor Name
•	Department Name
•	Total Appointments
•	Total Revenue
________________________________________
Question 5
Find all pending appointments.
Expected output should show:
•	Appointment ID
•	Patient Name
•	Doctor Name
•	Department Name
•	Appointment Date
•	Appointment Time
________________________________________
Question 6
Find unpaid bills.
Expected output should show:
•	Patient Name
•	Doctor Name
•	Department Name
•	Bill Amount
•	Payment Status
________________________________________
Question 7
Find city-wise patient visits.
Expected output should show:
•	Patient City
•	Total Number of Appointments
________________________________________
Final Assignment
Create another view named:
V_DEPARTMENT_DAILY_REVENUE
This view should summarize daily OPD revenue department-wise.
The view should include:
Field Name	Description
Appointment Date	Date of appointment
Department Name	Hospital department
Total Appointments	Number of completed appointments
Total Paid Bills	Number of paid bills
Total Revenue	Revenue generated
The view should only consider:
•	Completed appointments
•	Paid bills
________________________________________
Expected Learning Outcome
After completing this hands-on exercise, students should understand:
1.	How to design a database schema for a real-world business case.
2.	How to create multiple related tables.
3.	How to use primary key and foreign key relationships.
4.	How to store transactional data.
5.	How to join multiple tables.
6.	How to create a SQL view for reporting.
7.	How to use views for business analytics.
8.	How to answer business questions using structured SQL data.